In [ ]:
import numpy as np
import pandas as pd
import random
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import AffinityPropagation
from sklearn.metrics import pairwise_distances

# --- 1. GENERATE SYNTHETIC DATA ---
print("--- Generating Synthetic Plasmids ---")

def generate_dna(length):
    return "".join(random.choices("ACGT", k=length))

# Set fixed seed for reproducibility
random.seed(42)

# Create 7 Distinct Barcodes (20bp) and 7 Distinct Inserts (100bp)
barcodes = {f"B{i}": generate_dna(20) for i in range(1, 8)}
inserts  = {f"I{i}": generate_dna(100) for i in range(1, 8)}

# Construct the Challenge Dataset
# We create 10 'reads' for each combo to simulate depth
reads_data = []

combos = [
    # --- THE TRAPS ---
    ("B1", "I1"), # Barcode 1 with Insert 1
    ("B1", "I2"), # Barcode 1 with Insert 2 (Same Barcode, Diff Insert!)
    
    ("B2", "I3"), # Barcode 2 with Insert 3
    ("B3", "I3"), # Barcode 3 with Insert 3 (Same Insert, Diff Barcode!)
    
    # --- THE CONTROLS (Normal 1:1) ---
    ("B4", "I4"),
    ("B5", "I5"),
    ("B6", "I6"),
    ("B7", "I7"),
]

for bc_name, ins_name in combos:
    bc_seq = barcodes[bc_name]
    ins_seq = inserts[ins_name]
    
    # Create 5 identical copies (reads) for each combo
    for i in range(5):
        reads_data.append({
            "Label": f"{bc_name}-{ins_name}", # Ground Truth Label
            "Barcode_Seq": bc_seq,
            "Insert_Seq": ins_seq
        })

df = pd.DataFrame(reads_data)
print(f"Created {len(df)} total reads representing {len(combos)} unique biological constructs.")
print("The 'Traps' are hidden in the data. Starting Clustering...\n")


# --- 2. COMPUTE DUAL MATRICES ---

def get_similarity_matrix(sequences, k=5):
    # 1. Vectorize (Keep it sparse!)
    vectorizer = CountVectorizer(analyzer='char', ngram_range=(k, k), binary=True)
    X = vectorizer.fit_transform(sequences)
    
    # 2. Calculate Intersection (Dot Product)
    # result is a sparse matrix (csr_matrix)
    intersection_sparse = X.dot(X.T)
    
    # --- THE FIX ---
    # Convert to a standard numpy ARRAY immediately.
    # This prevents 'np.matrix' creation during the subtraction step later.
    intersection = np.asarray(intersection_sparse.todense())
    
    # 3. Calculate Union
    # Union = |A| + |B| - Intersection
    cardinality = X.getnnz(axis=1) # Number of k-mers per read
    
    # Broadcasting: Column + Row - Matrix
    # Since 'intersection' is now a standard array, this math is safe.
    union = cardinality[:, None] + cardinality[None, :] - intersection
    
    # 4. Compute Similarity
    with np.errstate(divide='ignore', invalid='ignore'):
        similarity = intersection / union
        
    # Fix division by zero (0/0) -> 0
    similarity[np.isnan(similarity)] = 0.0
    
    return similarity

# Matrix A: Barcode Similarity
# We use k=3 for barcodes since they are short (20bp)
sim_barcode = get_similarity_matrix(df['Barcode_Seq'], k=3)

# Matrix B: Insert Similarity
# We use k=5 for inserts since they are longer (100bp)
sim_insert = get_similarity_matrix(df['Insert_Seq'], k=5)


# --- 3. FUSE MATRICES (EQUAL WEIGHTING) ---

# This is the secret sauce. 
# If Barcodes are identical (1.0) but Inserts are different (0.0), score is 0.5.
fused_similarity = (0.5 * sim_barcode) + (0.5 * sim_insert)


# --- 4. CLUSTERING ---

# Damping=0.9 helps AP converge when you have perfect clusters
af = AffinityPropagation(affinity='precomputed', damping=0.9, random_state=42)

# --- THE FIX: ADD JITTER ---
# Add tiny random noise to break the symmetry of identical reads
jitter = np.random.normal(0, 1e-5, fused_similarity.shape)
fused_similarity += jitter

# Ensure the matrix remains symmetric (Similarity A->B must equal B->A)
fused_similarity = (fused_similarity + fused_similarity.T) / 2

# Also ensure diagonal is 1.0 (Self-similarity)
np.fill_diagonal(fused_similarity, 1.0)

# --- CLUSTERING ---
# Lower damping to 0.5 (default) allows faster convergence now that ties are broken
af = AffinityPropagation(affinity='precomputed', damping=0.5, max_iter=1000, random_state=42)
af.fit(fused_similarity)


labels = af.labels_
df['Cluster_ID'] = labels


# --- 5. VALIDATION ---

print("--- Clustering Results ---")
print(f"Algorithm found {len(set(labels))} clusters (Expected: {len(combos)})")
print("\nDetailed Breakdown:")

# Group by Cluster ID to see what fell into each bin
for cid in sorted(set(labels)):
    cluster_content = df[df['Cluster_ID'] == cid]
    
    # Get the Ground Truth labels inside this cluster
    found_types = cluster_content['Label'].unique()
    size = len(cluster_content)
    
    # Check if the cluster is pure (only one ground truth type)
    status = "✅ PURE" if len(found_types) == 1 else "❌ MIXED"
    
    print(f"Cluster {cid} (n={size}): {found_types} {status}")

print("\n--- Edge Case Analysis ---")
# Check the specific traps
trap_b1 = df[df['Label'].str.contains("B1")]
trap_i3 = df[df['Label'].str.contains("I3")]

print(f"Reads with Barcode B1 were assigned to clusters: {trap_b1['Cluster_ID'].unique()}")
print(f"Reads with Insert I3 were assigned to clusters: {trap_i3['Cluster_ID'].unique()}")

if len(trap_b1['Cluster_ID'].unique()) == 2:
    print("SUCCESS: The algorithm successfully split the Shared Barcode (B1).")
else:
    print("FAILURE: The Shared Barcode was merged.")

if len(trap_i3['Cluster_ID'].unique()) == 2:
    print("SUCCESS: The algorithm successfully split the Shared Insert (I3).")
else:
    print("FAILURE: The Shared Insert was merged.")

In [ ]:
import numpy as np
import pandas as pd
import random
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.cluster import AffinityPropagation
from sklearn.pipeline import make_pipeline

# --- 1. DATA GENERATION (The same challenge dataset) ---
# 7 Barcodes, 7 Inserts. 
# Challenge 1: B1 is on I1 and I2. 
# Challenge 2: I3 is with B2 and B3.
# Total Expected Clusters: 8

random.seed(42)
np.random.seed(42)

def generate_dna(length): return "".join(random.choices("ACGT", k=length))
barcodes = {f"B{i}": generate_dna(20) for i in range(1, 8)}
inserts  = {f"I{i}": generate_dna(100) for i in range(1, 8)}

combos = [("B1", "I1"), ("B1", "I2"), ("B2", "I3"), ("B3", "I3"), 
          ("B4", "I4"), ("B5", "I5"), ("B6", "I6"), ("B7", "I7")]

data = []
for bc, ins in combos:
    # 5 reads per combo to test convergence
    for i in range(5):
        data.append({"Label": f"{bc}-{ins}", "Barcode": barcodes[bc], "Insert": inserts[ins]})
df = pd.DataFrame(data)

# --- 2. THE EMBEDDING PIPELINE ---

def create_embedding(sequences, k, n_components=10):
    """
    Turns sequences into a dense, normalized embedding vector.
    """
    # Step A: K-mer counting (Sparse High-Dimensional)
    # Using TF-IDF logic (binary=False) here can sometimes help downweight common repeats,
    # but binary=True is safer for pure identity.
    vectorizer = CountVectorizer(analyzer='char', ngram_range=(k, k), binary=True)
    
    # Step B: Dimensionality Reduction (The "Embedding")
    # We reduce the huge sparse matrix down to 'n_components' dense features.
    # This captures the 'essence' of the sequence.
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    
    # Step C: L2 Normalization
    # CRITICAL: This ensures the vector has length 1.0. 
    # Without this, the Insert embedding might have huge values compared to Barcode.
    normalizer = Normalizer(norm='l2')
    
    pipeline = make_pipeline(vectorizer, svd, normalizer)
    return pipeline.fit_transform(sequences)

# --- 3. EXECUTION ---

print("Generating Embeddings...")

# Create Barcode Embedding (10 Dimensions)
# k=3 for fine-grained sensitivity
emb_barcode = create_embedding(df['Barcode'], k=3, n_components=10)

# Create Insert Embedding (10 Dimensions)
# k=5 for robustness
emb_insert = create_embedding(df['Insert'], k=5, n_components=10)

print(f"Barcode Shape: {emb_barcode.shape}") # (40, 10)
print(f"Insert Shape:  {emb_insert.shape}")  # (40, 10)


# --- 4. CONCATENATION (LATE FUSION) ---

# Glue them together. 
# Result is a 20-dimensional vector where the first 10 dims are Barcode and last 10 are Insert.
final_embedding = np.hstack([emb_barcode, emb_insert])

print(f"Fused Shape:   {final_embedding.shape}") # (40, 20)


# --- 5. CLUSTERING ---

# Since we now have a clean, dense coordinate space (Euclidean geometry), 
# we don't need precomputed matrices. We can feed the vectors directly to AP.
# We add slight noise (jitter) to break perfect symmetries in synthetic data.
jitter = np.random.normal(0, 1e-4, final_embedding.shape)
final_embedding_noisy = final_embedding + jitter

print("Clustering on Fused Embeddings...")
af = AffinityPropagation(damping=0.8, random_state=42)
af.fit(final_embedding_noisy)

df['Cluster'] = af.labels_


# --- 6. RESULTS ---
print(f"\nFound {len(set(af.labels_))} clusters (Expected: 8).")

# Verify the "Traps"
b1_reads = df[df['Label'].str.contains("B1")]
i3_reads = df[df['Label'].str.contains("I3")]

print(f"\nBarcode B1 (Attached to I1 & I2) split into clusters: {b1_reads['Cluster'].unique()}")
print(f"Insert I3 (Attached to B2 & B3) split into clusters: {i3_reads['Cluster'].unique()}")

if len(b1_reads['Cluster'].unique()) == 2 and len(i3_reads['Cluster'].unique()) == 2:
    print("\nSUCCESS: Both traps were successfully resolved using Late Fusion.")

In [3]:
import numpy as np
import pandas as pd
import difflib
import random
import time
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.metrics import pairwise_distances

# --- 1. CHAOTIC DATA GENERATION ---
random.seed(42)
np.random.seed(42)

def mutate_sequence(seq, error_rate=0.2):
    """Simulates Nanopore noise: Indels and Substitutions."""
    seq_list = list(seq)
    new_seq = []
    for base in seq_list:
        if random.random() < error_rate:
            type_r = random.random()
            if type_r < 0.33: new_seq.append(random.choice("ACGT")) # Sub
            elif type_r < 0.66: pass # Del
            else: # Ins
                new_seq.append(base)
                new_seq.append(random.choice("ACGT"))
        else:
            new_seq.append(base)
    return "".join(new_seq)

def generate_dna(length): 
    return "".join(random.choices("ACGT", k=length))

print("--- Generating 1000 Biological Entities (Noisy & Unbalanced) ---")

n_items = 1000
pool_barcodes = [generate_dna(20) for _ in range(n_items)]
pool_inserts  = [generate_dna(150) for _ in range(n_items)]

# Define Truth & Traps
combos = []
combos.append({"Label": "Trap_SharedBC_A", "BC": pool_barcodes[0], "Ins": pool_inserts[0]})
combos.append({"Label": "Trap_SharedBC_B", "BC": pool_barcodes[0], "Ins": pool_inserts[1]})
combos.append({"Label": "Trap_SharedIns_A", "BC": pool_barcodes[1], "Ins": pool_inserts[2]})
combos.append({"Label": "Trap_SharedIns_B", "BC": pool_barcodes[2], "Ins": pool_inserts[2]})

for i in range(3, n_items):
    combos.append({"Label": f"Standard_{i}", "BC": pool_barcodes[i], "Ins": pool_inserts[i]})

# Generate Reads
data = []
for combo in combos:
    # Random abundance (1 to 10 reads)
    n_reads = random.choices([1, 2, 3, 5, 10], weights=[0.2, 0.4, 0.2, 0.1, 0.1])[0]
    for _ in range(n_reads):
        data.append({
            "Label": combo["Label"], 
            "Barcode": mutate_sequence(combo["BC"]), 
            "Insert": mutate_sequence(combo["Ins"])
        })

df = pd.DataFrame(data)
total_reads = len(df)
print(f"Total Dataset: {total_reads} reads.")


# --- 2. BARCODE LANDMARKS (Robust Linear Complexity) ---
print(f"\n[1/5] Computing Barcode Landmarks...")

n_landmarks = 100 
landmark_indices = np.random.choice(total_reads, n_landmarks, replace=False)
landmarks = [df.iloc[i]['Barcode'] for i in landmark_indices]

landmark_vectors = np.zeros((total_reads, n_landmarks))

for i, seq in enumerate(df['Barcode']):
    for j, mark in enumerate(landmarks):
        sm = difflib.SequenceMatcher(None, seq, mark, autojunk=False)
        landmark_vectors[i, j] = 1.0 - sm.ratio()

dist_barcode = pairwise_distances(landmark_vectors, metric='cosine')


# --- 3. INSERT DISTANCE (SVD) ---
print("\n[2/5] Computing Insert Distances...")
vectorizer = CountVectorizer(analyzer='char', ngram_range=(4, 4), binary=True)
svd = TruncatedSVD(n_components=40, random_state=42)
normalizer = Normalizer(norm='l2')

insert_vectors = normalizer.fit_transform(svd.fit_transform(vectorizer.fit_transform(df['Insert'])))
dist_insert = pairwise_distances(insert_vectors, metric='cosine')


# --- 4. STRICT FUSION (THE FIX) ---
print("\n[3/5] Fusing with Veto Logic (Max Distance)...")

def scale_by_mean(matrix):
    mask = ~np.eye(matrix.shape[0], dtype=bool)
    mu = matrix[mask].mean()
    return matrix / mu if mu > 1e-9 else matrix

# Normalize
norm_bc = scale_by_mean(dist_barcode)
norm_ins = scale_by_mean(dist_insert)

# Weighting: Boost Barcode slightly (1.2x) as it is the primary ID
norm_bc = norm_bc * 1.2

# THE FIX: Use Maximum instead of Average
# If EITHER signal says "Different", they are different.
final_matrix = np.maximum(norm_bc, norm_ins)


# --- 5. AUTO-TUNING CLUSTERING ---
TARGET_CLUSTERS = 1000   
WINDOW_PCT = 0.2         

print(f"\n[4/5] Auto-Tuning (Target: ~{TARGET_CLUSTERS})...")

# Build Hierarchy
condensed_matrix = squareform(final_matrix)
Z = linkage(condensed_matrix, method='average')

# Analyze Stability
merge_distances = Z[:, 2]
n_samples = final_matrix.shape[0]

min_k = int(TARGET_CLUSTERS * (1 - WINDOW_PCT))
max_k = int(TARGET_CLUSTERS * (1 + WINDOW_PCT))

best_k = TARGET_CLUSTERS
max_lifetime = -1.0
best_threshold = 0.0

for k in range(min_k, max_k + 1):
    idx = n_samples - k
    if idx >= len(merge_distances): continue
    
    current_dist = merge_distances[idx-1]
    next_dist = merge_distances[idx]
    lifetime = next_dist - current_dist
    
    if lifetime > max_lifetime:
        max_lifetime = lifetime
        best_k = k
        best_threshold = current_dist + (lifetime / 2)

print(f"   Best K: {best_k} | Stability: {max_lifetime:.4f} | Threshold: {best_threshold:.4f}")

# Apply Cut
labels = fcluster(Z, t=best_threshold, criterion='distance')
df['Cluster'] = labels - 1 


# --- 6. FORENSIC VALIDATION ---
print("\n[5/5] Final Validation...")

# Check Trap 1 (Shared Barcode)
t1 = df[df['Label'].str.contains("Trap_SharedBC")]
t1_clusters = t1['Cluster'].unique()
print(f"   Trap 1 (Shared BC) -> Clusters: {len(t1_clusters)} {t1_clusters}")
if len(t1_clusters) == 2: print("   ✅ SUCCESS (Insert forced split)")
else: print("   ❌ FAIL")

# Check Trap 2 (Shared Insert)
t2 = df[df['Label'].str.contains("Trap_SharedIns")]
t2_clusters = t2['Cluster'].unique()
print(f"   Trap 2 (Shared Ins) -> Clusters: {len(t2_clusters)} {t2_clusters}")
if len(t2_clusters) == 2: print("   ✅ SUCCESS (Barcode forced split)")
else: print("   ❌ FAIL")

# Check Global Fragmentation
n_final = len(set(df['Cluster']))
print(f"\n   Final Cluster Count: {n_final} (Target: {len(combos)})")
if n_final >= len(combos) and n_final < len(combos) * 1.05:
    print("   🏆 RESULT: Excellent Precision.")
else:
    print("   ⚠️ RESULT: Some fragmentation/merging occurred.")

--- Generating 1000 Biological Entities (Noisy & Unbalanced) ---
Total Dataset: 3073 reads.

[1/5] Computing Barcode Landmarks...

[2/5] Computing Insert Distances...

[3/5] Fusing with Veto Logic (Max Distance)...

[4/5] Auto-Tuning (Target: ~1000)...
   Best K: 866 | Stability: 0.0015 | Threshold: 0.9388

[5/5] Final Validation...
   Trap 1 (Shared BC) -> Clusters: 4 [553 496 648 568]
   ❌ FAIL
   Trap 2 (Shared Ins) -> Clusters: 4 [268 432 515 328]
   ❌ FAIL

   Final Cluster Count: 866 (Target: 1001)
   ⚠️ RESULT: Some fragmentation/merging occurred.


In [ ]:
# --- 7. FINAL TRAP VALIDATION ---

print("\n--- 🔬 Forensic Analysis of Decoys (Traps) ---")

# 1. Define the Traps we injected
# Trap 1: Shared Barcode (Same BC, Different Insert)
# Ground Truth: Should be 2 distinct clusters
trap_bc_reads = df[df['Label'].str.contains("Trap_SharedBC")]
trap_bc_clusters = trap_bc_reads['Cluster'].unique()

# Trap 2: Shared Insert (Different BC, Same Insert)
# Ground Truth: Should be 2 distinct clusters
trap_ins_reads = df[df['Label'].str.contains("Trap_SharedIns")]
trap_ins_clusters = trap_ins_reads['Cluster'].unique()

# 2. Check Trap 1
print(f"\n[Trap 1] Shared Barcode Test:")
print(f"   Reads involved: {len(trap_bc_reads)}")
print(f"   Assigned to Clusters: {trap_bc_clusters}")

if len(trap_bc_clusters) == 2:
    print("   ✅ SUCCESS: The algorithm forced a split based on Insert differences.")
    # Optional: Check if they are roughly equal size
    c1_count = len(trap_bc_reads[trap_bc_reads['Cluster'] == trap_bc_clusters[0]])
    c2_count = len(trap_bc_reads[trap_bc_reads['Cluster'] == trap_bc_clusters[1]])
    print(f"      Split Balance: {c1_count} reads vs {c2_count} reads")
elif len(trap_bc_clusters) == 1:
    print("   ❌ FAIL: The algorithm merged them (Barcode dominated the signal).")
else:
    print(f"   ⚠️ WARNING: Fragmented into {len(trap_bc_clusters)} clusters (Over-split).")


# 3. Check Trap 2
print(f"\n[Trap 2] Shared Insert Test:")
print(f"   Reads involved: {len(trap_ins_reads)}")
print(f"   Assigned to Clusters: {trap_ins_clusters}")

if len(trap_ins_clusters) == 2:
    print("   ✅ SUCCESS: The algorithm forced a split based on Barcode differences.")
elif len(trap_ins_clusters) == 1:
    print("   ❌ FAIL: The algorithm merged them (Insert dominated the signal).")
else:
    print(f"   ⚠️ WARNING: Fragmented into {len(trap_ins_clusters)} clusters (Over-split).")


# 4. Check Controls (The "Standard" Plasmids)
# We check a random sample of standards to ensure they didn't accidentally split
print(f"\n[Control] Purity Check:")
standard_reads = df[df['Label'].str.contains("Standard")]
sample_standards = random.sample(list(standard_reads['Label'].unique()), 5)

for label in sample_standards:
    clusters = df[df['Label'] == label]['Cluster'].unique()
    status = "✅ Stable" if len(clusters) == 1 else f"❌ Fragmented ({len(clusters)} parts)"
    print(f"   {label}: {status}")

# 5. Global Summary
if len(trap_bc_clusters) == 2 and len(trap_ins_clusters) == 2:
    print("\n🏆 RESULT: ALGORITHM IS VALIDATED.")
    print("   It successfully balanced Barcode and Insert signals to distinguish all biological entities.")
else:
    print("\n💥 RESULT: ALGORITHM NEEDS TUNING.")

In [1]:
import numpy as np
import pandas as pd
import random
import time
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from scipy.stats import rankdata
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.metrics import pairwise_distances

# --- 1. CHAOTIC DATA GENERATION (Simulating Nanopore Noise) ---
print("--- Generating 1000 Biological Entities ---")
random.seed(42)
np.random.seed(42)

def mutate_sequence(seq, error_rate=0.03):
    """Simulates Nanopore noise: Indels (fatal for one-hot) and Subs."""
    seq_list = list(seq)
    new_seq = []
    for base in seq_list:
        if random.random() < error_rate:
            type_r = random.random()
            if type_r < 0.33: new_seq.append(random.choice("ACGT")) # Sub
            elif type_r < 0.66: pass # Del (Frame shift!)
            else: 
                new_seq.append(base)
                new_seq.append(random.choice("ACGT")) # Ins
        else:
            new_seq.append(base)
    return "".join(new_seq)

def generate_dna(length): 
    return "".join(random.choices("ACGT", k=length))

# Generate True Entities
n_items = 1000
pool_barcodes = [generate_dna(20) for _ in range(n_items)]
pool_inserts  = [generate_dna(150) for _ in range(n_items)]

# Define Truth & Traps
combos = []
# Trap 1: Same Barcode, Different Insert (Should be split)
combos.append({"Label": "Trap_SharedBC_A", "BC": pool_barcodes[0], "Ins": pool_inserts[0]})
combos.append({"Label": "Trap_SharedBC_B", "BC": pool_barcodes[0], "Ins": pool_inserts[1]})
# Trap 2: Different Barcode, Same Insert (Should be split)
combos.append({"Label": "Trap_SharedIns_A", "BC": pool_barcodes[1], "Ins": pool_inserts[2]})
combos.append({"Label": "Trap_SharedIns_B", "BC": pool_barcodes[2], "Ins": pool_inserts[2]})

for i in range(3, n_items):
    combos.append({"Label": f"Standard_{i}", "BC": pool_barcodes[i], "Ins": pool_inserts[i]})

# Generate Reads (1-3 reads per entity)
data = []
for combo in combos:
    n_reads = random.choices([1, 2, 3], weights=[0.2, 0.4, 0.4])[0]
    for _ in range(n_reads):
        data.append({
            "Label": combo["Label"], 
            "Barcode": mutate_sequence(combo["BC"]), 
            "Insert": mutate_sequence(combo["Ins"])
        })

df = pd.DataFrame(data)
print(f"Dataset Generated: {len(df)} reads.")


# --- 2. BARCODE VECTORIZATION (The Short K-mer Fix) ---
print("\n[1/4] Processing Barcodes (Shift-Invariant K-mers)...")

# CRITICAL: For short sequences (15-20nt) with indels:
# 1. ngram_range=(2, 3): Captures Bigrams (robust) AND Trigrams (specific).
# 2. No SVD: The feature space is small enough (~300 dims), SVD loses too much info here.
bc_vectorizer = CountVectorizer(
    analyzer='char', 
    ngram_range=(2, 3), # <--- The "Short K-mer" Logic
    binary=False        # Count frequency (e.g., 'AA' appearing twice matters in short seqs)
)

bc_vectors = bc_vectorizer.fit_transform(df['Barcode'])

# Normalize: Critical because read lengths vary due to indels
normalizer = Normalizer(norm='l2')
bc_vectors = normalizer.transform(bc_vectors.astype(float))


# --- 3. INSERT VECTORIZATION (Standard SVD) ---
print("[2/4] Processing Inserts (LSA/SVD)...")

# Inserts are long (150bp). We use larger K (4) and SVD to reduce noise.
ins_vectorizer = CountVectorizer(analyzer='char', ngram_range=(4, 4), binary=False)
svd = TruncatedSVD(n_components=50, random_state=42)

ins_raw = ins_vectorizer.fit_transform(df['Insert'])
ins_vectors = normalizer.transform(svd.fit_transform(ins_raw))


# --- 4. ROBUST FUSION (Rank Norm + Maximum) ---
print("[3/4] Fusing Signals...")

# Calculate Raw Cosine Distances
dist_bc = pairwise_distances(bc_vectors, metric='cosine')
dist_ins = pairwise_distances(ins_vectors, metric='cosine')

def rank_normalize(matrix):
    """
    Converts distances to Percentiles (0.0 to 1.0).
    Solves the issue where Barcode distance 0.2 != Insert distance 0.2
    """
    # rankdata flattens the array, ranks them, then we reshape back
    ranked = rankdata(matrix)
    return (ranked.reshape(matrix.shape) - 1) / (ranked.max() - 1)

# Apply Rank Normalization
norm_bc = rank_normalize(dist_bc)
norm_ins = rank_normalize(dist_ins)

# Weighting: We trust Barcodes slightly more (1.0 vs 0.9)
norm_bc = norm_bc * 1.0
norm_ins = norm_ins * 0.9 

# STRICT FUSION: Use Maximum
# If Barcodes say "Different" (1.0) but Inserts say "Same" (0.0) -> Result is "Different" (1.0)
final_matrix = np.maximum(norm_bc, norm_ins)

np.fill_diagonal(final_matrix, 0)


# --- 5. CLUSTERING & AUTO-TUNING ---
print(f"[4/4] Auto-Tuning Clusters (Target: ~{n_items})...")

condensed_matrix = squareform(final_matrix)
Z = linkage(condensed_matrix, method='average')

# Stability Analysis
merge_distances = Z[:, 2]
n_samples = final_matrix.shape[0]
target_k = n_items
window = 200 # Search window

best_k = target_k
max_lifetime = -1.0
best_threshold = 0.0

start_k = max(2, target_k - window)
end_k = min(n_samples - 1, target_k + window)

for k in range(start_k, end_k):
    idx = n_samples - k
    if idx >= len(merge_distances): continue
    
    # "Lifetime" is how much distance grows before the next merge happens
    lifetime = merge_distances[idx] - merge_distances[idx-1]
    
    if lifetime > max_lifetime:
        max_lifetime = lifetime
        best_k = k
        # Cut in the middle of the stable zone
        best_threshold = merge_distances[idx-1] + (lifetime / 2)

print(f"   Stability Optimized K: {best_k} | Threshold: {best_threshold:.4f}")

# Apply Cluster Labels
labels = fcluster(Z, t=best_threshold, criterion='distance')
df['Cluster'] = labels


# --- 6. VALIDATION ---
print("\n--- Final Forensic Report ---")

# Check Trap 1: Same Barcode, Different Insert
t1 = df[df['Label'].str.contains("Trap_SharedBC")]
t1_clusters = t1['Cluster'].unique()
print(f"Trap 1 (Shared BC, Diff Ins): Found in {len(t1_clusters)} clusters (Expected 2).")
if len(t1_clusters) == 2: print("   ✅ SUCCESS: Inserts forced a split.")
else: print("   ❌ FAIL: Merged incorrectly.")

# Check Trap 2: Different Barcode, Same Insert
t2 = df[df['Label'].str.contains("Trap_SharedIns")]
t2_clusters = t2['Cluster'].unique()
print(f"Trap 2 (Diff BC, Shared Ins): Found in {len(t2_clusters)} clusters (Expected 2).")
if len(t2_clusters) == 2: print("   ✅ SUCCESS: Barcodes forced a split.")
else: print("   ❌ FAIL: Merged incorrectly.")

# Global fragmentation check
n_final = len(set(df['Cluster']))
print(f"\nTotal Clusters Found: {n_final}")
print(f"Total True Entities:  {len(combos)}")
accuracy = 1.0 - (abs(n_final - len(combos)) / len(combos))
print(f"Estimated Accuracy:   {accuracy*100:.1f}%")

--- Generating 1000 Biological Entities ---
Dataset Generated: 2238 reads.

[1/4] Processing Barcodes (Shift-Invariant K-mers)...
[2/4] Processing Inserts (LSA/SVD)...
[3/4] Fusing Signals...
[4/4] Auto-Tuning Clusters (Target: ~1000)...
   Stability Optimized K: 999 | Threshold: 0.0071

--- Final Forensic Report ---
Trap 1 (Shared BC, Diff Ins): Found in 2 clusters (Expected 2).
   ✅ SUCCESS: Inserts forced a split.
Trap 2 (Diff BC, Shared Ins): Found in 2 clusters (Expected 2).
   ✅ SUCCESS: Barcodes forced a split.

Total Clusters Found: 999
Total True Entities:  1001
Estimated Accuracy:   99.8%


In [ ]:
# --- 7. FINAL TRAP VALIDATION ---

print("\n--- 🔬 Forensic Analysis of Decoys (Traps) ---")

# 1. Define the Traps we injected
# Trap 1: Shared Barcode (Same BC, Different Insert)
# Ground Truth: Should be 2 distinct clusters
trap_bc_reads = df[df['Label'].str.contains("Trap_SharedBC")]
trap_bc_clusters = trap_bc_reads['Cluster'].unique()

# Trap 2: Shared Insert (Different BC, Same Insert)
# Ground Truth: Should be 2 distinct clusters
trap_ins_reads = df[df['Label'].str.contains("Trap_SharedIns")]
trap_ins_clusters = trap_ins_reads['Cluster'].unique()

# 2. Check Trap 1
print(f"\n[Trap 1] Shared Barcode Test:")
print(f"   Reads involved: {len(trap_bc_reads)}")
print(f"   Assigned to Clusters: {trap_bc_clusters}")

if len(trap_bc_clusters) == 2:
    print("   ✅ SUCCESS: The algorithm forced a split based on Insert differences.")
    # Optional: Check if they are roughly equal size
    c1_count = len(trap_bc_reads[trap_bc_reads['Cluster'] == trap_bc_clusters[0]])
    c2_count = len(trap_bc_reads[trap_bc_reads['Cluster'] == trap_bc_clusters[1]])
    print(f"      Split Balance: {c1_count} reads vs {c2_count} reads")
elif len(trap_bc_clusters) == 1:
    print("   ❌ FAIL: The algorithm merged them (Barcode dominated the signal).")
else:
    print(f"   ⚠️ WARNING: Fragmented into {len(trap_bc_clusters)} clusters (Over-split).")


# 3. Check Trap 2
print(f"\n[Trap 2] Shared Insert Test:")
print(f"   Reads involved: {len(trap_ins_reads)}")
print(f"   Assigned to Clusters: {trap_ins_clusters}")

if len(trap_ins_clusters) == 2:
    print("   ✅ SUCCESS: The algorithm forced a split based on Barcode differences.")
elif len(trap_ins_clusters) == 1:
    print("   ❌ FAIL: The algorithm merged them (Insert dominated the signal).")
else:
    print(f"   ⚠️ WARNING: Fragmented into {len(trap_ins_clusters)} clusters (Over-split).")


# 4. Check Controls (The "Standard" Plasmids)
# We check a random sample of standards to ensure they didn't accidentally split
print(f"\n[Control] Purity Check:")
standard_reads = df[df['Label'].str.contains("Standard")]
sample_standards = random.sample(list(standard_reads['Label'].unique()), 5)

for label in sample_standards:
    clusters = df[df['Label'] == label]['Cluster'].unique()
    status = "✅ Stable" if len(clusters) == 1 else f"❌ Fragmented ({len(clusters)} parts)"
    print(f"   {label}: {status}")

# 5. Global Summary
if len(trap_bc_clusters) == 2 and len(trap_ins_clusters) == 2:
    print("\n🏆 RESULT: ALGORITHM IS VALIDATED.")
    print("   It successfully balanced Barcode and Insert signals to distinguish all biological entities.")
else:
    print("\n💥 RESULT: ALGORITHM NEEDS TUNING.")